# Specimen (PMDCo): from data entry to RDF

This notebook shows how to describe a physical specimen (its name, mass, and
chemical composition) and convert that description into a standardised,
machine-readable RDF graph following the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.input.json
  │  your specimen name, mass, and element fractions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows key values from the graph
```

---

## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [ ]:
%pip install -q jsonata-python rdflib pyyaml pyshacl

In [ ]:
import sys, json, pathlib, yaml, jsonata, pyshacl, rdflib
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE       = pathlib.Path().resolve()          # docs/
SPECIMEN   = HERE.parent                       # schemas/specimen/PMDCo/
STORE_ROOT = SPECIMEN.parents[2]               # semantic-schemas/
CHEM_COMP  = STORE_ROOT / "schemas" / "chemical-composition" / "PMDCo"

---
## Step 1: Describe your specimen

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `specimen_name` | yes | A name or ID for the specimen |
| `elements` | yes | List of `{symbol, value, unit}` (same format as the chemical-composition schema) |
| `mass_value` | no | Mass of the specimen as a positive number |
| `mass_unit` | no | `"g"` (gram), `"kg"` (kilogram), or `"mg"` (milligram) |
| `specimen_id` | no | Custom IRI slug (auto-derived from `specimen_name` if omitted) |
| `comp_id` | no | Custom IRI slug for the composition node (auto-derived if omitted) |

In [ ]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

---
## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document, the intermediate
format that carries both the data and its ontology mapping.

The specimen schema is built on top of the
[chemical composition schema](../../chemical-composition/PMDCo/README.md):
the composition part of the output is produced by that schema's converter,
which is the authoritative source for element IRIs and node naming conventions.
The two results are then merged into a single document.

In [ ]:
# ── Specimen envelope (name, mass) ─────────────────────────────────────
specimen_expr = (SPECIMEN / "simplified" / "transform.jsonata").read_text()
specimen_doc  = jsonata.Jsonata(specimen_expr).evaluate(simplified)

In [ ]:
# ── Chemical composition (delegated to the composition schema's converter) ──
comp_input = {
    "material_name": simplified["specimen_name"],
    "material_id":   specimen_doc["id"],
    "elements":      simplified["elements"],
}
if "comp_id" in simplified:
    comp_input["comp_id"] = simplified["comp_id"]

comp_expr = (CHEM_COMP / "simplified" / "transform.jsonata").read_text()
comp_doc  = jsonata.Jsonata(comp_expr).evaluate(comp_input)

# The composition converter embeds the material as a labelled node (standalone use).
# Here the material is the specimen itself, so we replace it with the specimen IRI.
comp_doc["quality_of"] = specimen_doc["id"]

In [ ]:
# ── Merge into the final OO-LD document ───────────────────────────────
oold_doc = {**specimen_doc, "has_composition": comp_doc}

print(json.dumps(oold_doc, indent=2))

---
## Step 3: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [ ]:
context = yaml.safe_load(
    (SPECIMEN / "specs" / "schema.oold.yaml").read_text()
)["@context"]

g = rdflib.Dataset()
g.parse(
    data   = json.dumps({"@context": context, **oold_doc}),
    format = "json-ld",
)

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_specimen.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

---
## Step 4: Validate against the SHACL shapes

The graph is checked against two SHACL shape files: one for the specimen
(mass, label) and one for the chemical composition (element fractions,
proportions):

| Shape file | Validates |
|---|---|
| `specimen/PMDCo/specs/shape.ttl` | Specimen label, Mass value and unit |
| `chemical-composition/PMDCo/specs/shape.ttl` | ChemicalComposition, element fractions |

> `inference="rdfs"` is required because some shapes target superclasses
> (e.g. `Proportion`) and rely on RDFS reasoning to match their subtypes.

In [ ]:
shapes_graph = rdflib.Graph()
shapes_graph.parse(str(SPECIMEN  / "specs" / "shape.ttl"))
shapes_graph.parse(str(CHEM_COMP / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
    inference   = "rdfs",
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  ✗ {msg}" + (f"\n    property: {loc}" if loc else ""))

---
## Step 5: Inspect the graph

Quick checks to confirm the key values were captured correctly.

In [ ]:
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

BFO   = rdflib.Namespace("http://purl.obolibrary.org/obo/BFO_")
RO    = rdflib.Namespace("http://purl.obolibrary.org/obo/RO_")
PMDCO = rdflib.Namespace("https://w3id.org/pmd/co/")
OBI   = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
IAO   = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")

# The specimen has two has_quality links (mass + composition).
# Identify the Mass node by its RDF type.
specimen_iri = next(flat.subjects(rdflib.RDF.type, BFO["0000030"]))
label        = flat.value(specimen_iri, rdflib.RDFS.label)
mass_quality = next(
    q for q in flat.objects(specimen_iri, RO["0000086"])
    if (q, rdflib.RDF.type, PMDCO["PMD_0020133"]) in flat
)
mass_svs  = flat.value(mass_quality, PMDCO["PMD_0000077"])
mass_val  = flat.value(mass_svs, OBI["0001937"])
mass_unit = flat.value(mass_svs, IAO["0000039"])

print(f"Specimen : {specimen_iri}")
print(f"Label    : {label}")
print(f"Mass     : {mass_val} {str(mass_unit).rsplit('/', 1)[-1]}")

In [ ]:
SPARQL_ELEMENTS = """
PREFIX ro:    <http://purl.obolibrary.org/obo/RO_>
PREFIX pmdco: <https://w3id.org/pmd/co/>
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>

SELECT ?elemType ?fracValue ?unit
WHERE {
  ?spec a pmdco:PMD_0025002 ;
        ro:0002351 ?frac .
  ?frac a pmdco:PMD_0025997 ;
        obi:0001937 ?fracValue ;
        iao:0000039 ?unit ;
        pmdco:PMD_hasFractionElement ?elemNode .
  ?elemNode a ?elemType .
  FILTER(?elemType != pmdco:PMD_0025997)
}
ORDER BY DESC(?fracValue)
"""

rows = list(flat.query(SPARQL_ELEMENTS))
print(f"{'Element (PMDCo IRI fragment)':<22}  {'Value':>8}  Unit")
print("-" * 50)
for r in rows:
    frag = str(r.elemType).rsplit("/", 1)[-1]
    unit = str(r.unit).rsplit("/", 1)[-1]
    print(f"{frag:<22}  {float(r.fracValue):>8.3f}  {unit}")

---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with specimen name, mass, and element fractions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against SHACL shapes |
| 5 | Key values are extracted from the graph to confirm correctness |

To use your own specimen, edit `docs/example.input.json` and re-run all cells.
`mass_value` / `mass_unit` are optional; omit them if the mass is not known.

---

## Further reading

- [Chemical Composition — PMDCo](../../../chemical-composition/PMDCo/README.md): the composition sub-schema used here
- [OO-LD primer](../../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../../docs/schema-format.md): how to write your own schema